# NB38: MASTER Panel — Heterojen Stacking + Calibrate-then-Shift + Raw Meta-Learner

**Amaç:** NB37'nin başarısız stacking'ini (homojen GBT base → OOF ~0.9 korele) **3 farklı deney** ile aşmak:

| Deney | Açıklama |
|-------|----------|
| **Deney 1: Heterojen Stacking** | 5 farklı aile (LightGBM, RF, BalancedBagging, SmallMLP, DNN) + Isotonic kalibrasyon + LR meta |
| **Deney 2: Calibrate-then-Shift** | En iyi base/stack çıkışına Saerens kapalı-form prior-düzeltmesi (π_train=0.733 → π_test=0.20) |
| **Deney 3: Raw Probability Meta-Learner** | Aynı 5 base'in HAM OOF olasılıklarını (kalibrasyon YOK) CatBoost / LGBM / LR meta-learner'a besle |

**Protokol:** NB36/NB37 ile birebir aynı split (SEED=42, %80/20 stratified), M3 missing, aynı değerlendirme.

**Birincil metrik:** %80/20 bootstrap pathogenic-F1 (N=50, %95 CI).

**Çıktı:** `results/v21_master_diverse_stack_labelshift/`, `reports/NB38_master_diverse_stack_labelshift_report.pdf`

In [1]:
# Cell 1: Imports ve Setup
import sys, os, warnings, time, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy
from collections import OrderedDict

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
    PROJECT_ROOT = os.path.dirname(os.getcwd())
if not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
    PROJECT_ROOT = os.getcwd()
    while PROJECT_ROOT != '/' and not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

from config import SEED, TEST_SIZE, PROJECT_ROOT, REPORTS_DIR
from src.columns_real import (
    ID_COL, TARGET_COL, NON_FEATURE_COLS,
    AL_COLS, CAT_COLS, EK_COLS, AA_COLS, ALL_FEATURE_COLS,
    AL_HIGH_MISSING_COLS, AL_MISSINGNESS_LEAKAGE_RISK, AL_SAFE_COLS,
    CAT_POPULATION_COLS, CAT_GENOTYPE_COLS, CAT_REGION_COLS,
    get_constant_cols, get_duplicate_col_pairs, get_missing_mask_col_name,
    AA_ALPHABET, AA_UNKNOWN_TOKEN
)
from src.metrics import optimize_threshold, compute_all_metrics
from src.features import GRANTHAM, BLOSUM62
from src.focal_loss import FocalLoss

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, matthews_corrcoef
from sklearn.metrics import precision_score, recall_score
from imblearn.ensemble import BalancedBaggingClassifier

import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from fpdf import FPDF

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v21_master_diverse_stack_labelshift')
os.makedirs(RESULTS_DIR, exist_ok=True)

PI_TRAIN = 0.733
PI_TEST = 0.20
N_BOOTSTRAP = 50
N_FOLDS = 5

print(f"PyTorch device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SEED: {SEED}, TEST_SIZE: {TEST_SIZE}")
print(f"Results: {RESULTS_DIR}")

PyTorch device: cpu
PROJECT_ROOT: /Users/tefe/teknofest_model/teknofest_model
SEED: 42, TEST_SIZE: 0.2
Results: /Users/tefe/teknofest_model/teknofest_model/results/v21_master_diverse_stack_labelshift


In [2]:
# Cell 2: Veri Yükleme ve İlk Temizlik
MASTER_CSV = os.path.join(PROJECT_ROOT, 'data/real_data/YARISMA_TRAIN_MASTER.csv')
df = pd.read_csv(MASTER_CSV)

print(f"Veri yüklendi: {df.shape}")
print(f"Label dağılımı:\n{df[TARGET_COL].value_counts()}")
print(f"Eksiklik oranı: {df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100:.2f}%")

Veri yüklendi: (2931, 353)
Label dağılımı:
Label
1    2149
0     782
Name: count, dtype: int64
Eksiklik oranı: 54.94%


In [3]:
# Cell 3: Stratified Split (NB37 ile birebir aynı)
feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape} (pos={y_train.sum()}, neg={(y_train==0).sum()})")
print(f"Test:  {X_test.shape} (pos={y_test.sum()}, neg={(y_test==0).sum()})")
print(f"Train prior (pathogenic): {y_train.mean():.4f}")

Train: (2344, 351) (pos=1719, neg=625)
Test:  (587, 351) (pos=430, neg=157)
Train prior (pathogenic): 0.7334


In [4]:
# Cell 4: Sabit ve Özdeş Sütun Temizliği (train üzerinde tespit)
const_cols = get_constant_cols(X_train)
dup_col_pairs = get_duplicate_col_pairs(X_train)
dup_cols = set()
for c1, c2 in dup_col_pairs:
    dup_cols.add(c2)

drop_cols = list(set(const_cols + list(dup_cols)))
if 'CAT_6' in X_train.columns and 'CAT_6' not in drop_cols:
    drop_cols.append('CAT_6')

feature_cols = [c for c in feature_cols if c not in drop_cols]
X_train = X_train[feature_cols].copy()
X_test = X_test[feature_cols].copy()

print(f"Sabit sütun: {len(const_cols)}")
print(f"Özdeş çift: {len(dup_col_pairs)} → {len(dup_cols)} drop")
print(f"Toplam drop: {len(drop_cols)} | Kalan feature: {X_train.shape[1]}")

Sabit sütun: 57
Özdeş çift: 583 → 58 drop
Toplam drop: 64 | Kalan feature: 287


In [5]:
# Cell 5: M3 Missing Stratejisi + Medyan Imputation
missing_threshold = 0.50
missing_mask_cols = {}

for col in feature_cols:
    miss_ratio = X_train[col].isnull().sum() / len(X_train)
    if miss_ratio > missing_threshold:
        mask_col_name = get_missing_mask_col_name(col)
        missing_mask_cols[col] = mask_col_name

print(f">%50 missing sütun: {len(missing_mask_cols)}")

imputer = SimpleImputer(strategy='median')
X_train_m3 = X_train.copy()
X_test_m3 = X_test.copy()

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    X_train_m3[numeric_cols] = imputer.fit_transform(X_train[numeric_cols])
    X_test_m3[numeric_cols] = imputer.transform(X_test[numeric_cols])

for orig_col, mask_col in missing_mask_cols.items():
    X_train_m3[mask_col] = X_train[orig_col].isnull().astype(int)
    X_test_m3[mask_col] = X_test[orig_col].isnull().astype(int)

print(f"Features (flags dahil): {X_train_m3.shape[1]}")

>%50 missing sütun: 139
Features (flags dahil): 426


In [6]:
# Cell 6: Değerlendirme Yardımcı Fonksiyonları (NB36/NB37 birebir)

def optimize_threshold_8020(y_true, y_prob, n_bootstrap=N_BOOTSTRAP, target_pos_rate=0.20):
    """%80/20 dağılımda F1-max ve MCC-max threshold seç."""
    best_thrs_f1, best_thrs_mcc = [], []
    rng = np.random.RandomState(SEED)
    idx_pos = np.where(y_true == 1)[0]
    idx_neg = np.where(y_true == 0)[0]
    for _ in range(n_bootstrap):
        n_neg = len(idx_neg)
        n_pos_target = max(1, int(n_neg * target_pos_rate / (1 - target_pos_rate)))
        if n_pos_target > len(idx_pos):
            n_pos_target = len(idx_pos)
        sel_pos = rng.choice(idx_pos, size=n_pos_target, replace=True)
        sel_neg = rng.choice(idx_neg, size=n_neg, replace=True)
        sel = np.concatenate([sel_pos, sel_neg])
        y_sel = y_true.values[sel] if hasattr(y_true, 'values') else y_true[sel]
        p_sel = y_prob[sel]
        best_f1, best_t_f1 = 0, 0.5
        best_mcc, best_t_mcc = -1, 0.5
        for thr in np.arange(0.10, 0.90, 0.01):
            preds = (p_sel >= thr).astype(int)
            f1 = f1_score(y_sel, preds, zero_division=0)
            mcc = matthews_corrcoef(y_sel, preds)
            if f1 > best_f1:
                best_f1, best_t_f1 = f1, thr
            if mcc > best_mcc:
                best_mcc, best_t_mcc = mcc, thr
        best_thrs_f1.append(best_t_f1)
        best_thrs_mcc.append(best_t_mcc)
    return np.median(best_thrs_f1), np.median(best_thrs_mcc)


def bootstrap_8020_eval(y_true, y_prob, threshold, n_bootstrap=N_BOOTSTRAP, target_pos_rate=0.20):
    """Bootstrap %80/20 dağılımda metrik hesapla."""
    rng = np.random.RandomState(SEED + 1)
    idx_pos = np.where(y_true == 1)[0]
    idx_neg = np.where(y_true == 0)[0]
    f1s, mccs, precs, recs = [], [], [], []
    for _ in range(n_bootstrap):
        n_neg = len(idx_neg)
        n_pos_target = max(1, int(n_neg * target_pos_rate / (1 - target_pos_rate)))
        if n_pos_target > len(idx_pos):
            n_pos_target = len(idx_pos)
        sel_pos = rng.choice(idx_pos, size=n_pos_target, replace=True)
        sel_neg = rng.choice(idx_neg, size=n_neg, replace=True)
        sel = np.concatenate([sel_pos, sel_neg])
        y_sel = y_true.values[sel] if hasattr(y_true, 'values') else y_true[sel]
        p_sel = y_prob[sel]
        preds = (p_sel >= threshold).astype(int)
        f1s.append(f1_score(y_sel, preds, zero_division=0))
        mccs.append(matthews_corrcoef(y_sel, preds))
        precs.append(precision_score(y_sel, preds, zero_division=0))
        recs.append(recall_score(y_sel, preds, zero_division=0))
    return {
        'f1_mean': float(np.mean(f1s)), 'f1_std': float(np.std(f1s)),
        'f1_ci_lo': float(np.percentile(f1s, 2.5)), 'f1_ci_hi': float(np.percentile(f1s, 97.5)),
        'f1_scores': f1s,
        'mcc_mean': float(np.mean(mccs)), 'mcc_std': float(np.std(mccs)),
        'prec_mean': float(np.mean(precs)), 'rec_mean': float(np.mean(recs)),
    }


def full_eval(name, y_true, y_prob, y_train_for_thr=None, proba_train_for_thr=None, verbose=True):
    """Tam değerlendirme: 3 threshold modu + %50/50 + %80/20 bootstrap.
    
    Threshold seçimi y_train_for_thr + proba_train_for_thr üzerinde yapılır (train OOF).
    Eğer verilmezse y_true + y_prob üzerinde seçilir (eski davranış).
    """
    y_thr = y_train_for_thr if y_train_for_thr is not None else y_true
    p_thr = proba_train_for_thr if proba_train_for_thr is not None else y_prob
    thr_raw, f1_raw = optimize_threshold(y_thr, p_thr)
    thr_8020, thr_mcc = optimize_threshold_8020(y_thr, p_thr)
    y_pred = (y_prob >= thr_8020).astype(int)
    m = compute_all_metrics(y_true, y_pred, y_prob)
    bs = bootstrap_8020_eval(y_true, y_prob, thr_8020)
    if verbose:
        print(f"  [{name}] thr_8020={thr_8020:.3f} | F1_5050={m['f1']:.4f} | "
              f"F1_8020={bs['f1_mean']:.4f} [{bs['f1_ci_lo']:.3f}-{bs['f1_ci_hi']:.3f}]")
    return {
        'name': name, 'thr_raw': thr_raw, 'thr_8020': thr_8020, 'thr_mcc': thr_mcc,
        'f1_5050': m['f1'], 'mcc_5050': m['mcc'], 'auc_roc': m['auc_roc'], 'auc_pr': m['auc_pr'],
        'prec_5050': m['precision'], 'rec_5050': m['recall'],
        'f1_8020': bs['f1_mean'], 'f1_8020_std': bs['f1_std'],
        'f1_8020_ci_lo': bs['f1_ci_lo'], 'f1_8020_ci_hi': bs['f1_ci_hi'],
        'f1_8020_scores': bs['f1_scores'],
        'mcc_8020': bs['mcc_mean'], 'prec_8020': bs['prec_mean'], 'rec_8020': bs['rec_mean'],
        'proba': y_prob,
    }


print("Değerlendirme fonksiyonları hazır.")

Değerlendirme fonksiyonları hazır.


In [7]:
# Cell 7: Encoding + Feature View Hazırlığı

def label_encode_cats(X_train, X_test):
    """Kategorik sütunları LabelEncode (train fit)."""
    Xtr, Xte = X_train.copy(), X_test.copy()
    cat_like = [c for c in Xtr.columns
                if Xtr[c].dtype == object or str(Xtr[c].dtype) == 'category']
    for col in cat_like:
        le = LabelEncoder()
        tr_vals = Xtr[col].astype('object').where(Xtr[col].notna(), '__NA__').astype(str)
        fit_vals = pd.concat([tr_vals, pd.Series(['__NA__'])], ignore_index=True)
        le.fit(fit_vals)
        Xtr[col] = le.transform(tr_vals)
        known = set(le.classes_)
        te_vals = Xte[col].astype('object').where(Xte[col].notna(), '__NA__').astype(str)
        te_vals = te_vals.map(lambda v, k=known: v if v in k else '__NA__')
        Xte[col] = le.transform(te_vals)
    for col in Xtr.columns:
        tr_num = pd.to_numeric(Xtr[col], errors='coerce')
        te_num = pd.to_numeric(Xte[col], errors='coerce')
        fill = tr_num.median()
        if pd.isna(fill):
            fill = 0.0
        Xtr[col] = tr_num.fillna(fill).astype(float)
        Xte[col] = te_num.fillna(fill).astype(float)
    return Xtr, Xte


# Tree-view: ham (NaN imputed) — LightGBM, RF, BalancedBagging
X_tr_tree, X_te_tree = label_encode_cats(X_train_m3, X_test_m3)

# NN-view: log1p + scale (her fold'da yeniden fit — burada global preview)
def make_nn_view(X_tr, X_te):
    """NN/DNN için log1p + StandardScaler (train fit)."""
    Xtr, Xte = X_tr.copy(), X_te.copy()
    num_cols = Xtr.select_dtypes(include=[np.number]).columns.tolist()
    for c in num_cols:
        Xtr[c] = np.log1p(np.abs(Xtr[c])) * np.sign(Xtr[c])
        Xte[c] = np.log1p(np.abs(Xte[c])) * np.sign(Xte[c])
    scaler = StandardScaler()
    Xtr[num_cols] = scaler.fit_transform(Xtr[num_cols])
    Xte[num_cols] = scaler.transform(Xte[num_cols])
    return Xtr, Xte, scaler

X_tr_nn, X_te_nn, nn_scaler_global = make_nn_view(X_tr_tree.copy(), X_te_tree.copy())

print(f"Tree-view: {X_tr_tree.shape}")
print(f"NN-view: {X_tr_nn.shape}")

Tree-view: (2344, 426)
NN-view: (2344, 426)


In [8]:
# Cell 8: Ortak OOF Fold Bölünmesi (tüm base'ler AYNI fold'ları paylaşır)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_INDICES = list(skf.split(X_tr_tree, y_train))

print(f"Fold sayısı: {len(FOLD_INDICES)}")
for i, (tr_idx, val_idx) in enumerate(FOLD_INDICES):
    y_tr_fold = y_train.iloc[tr_idx]
    y_val_fold = y_train.iloc[val_idx]
    print(f"  Fold {i}: train={len(tr_idx)} (pos={y_tr_fold.sum()}) | val={len(val_idx)} (pos={y_val_fold.sum()})")

Fold sayısı: 5
  Fold 0: train=1875 (pos=1375) | val=469 (pos=344)
  Fold 1: train=1875 (pos=1375) | val=469 (pos=344)
  Fold 2: train=1875 (pos=1375) | val=469 (pos=344)
  Fold 3: train=1875 (pos=1375) | val=469 (pos=344)
  Fold 4: train=1876 (pos=1376) | val=468 (pos=343)


In [9]:
# Cell 9: Base Model Eğitim Fonksiyonları (5 aile — ortak fold'lar)

def train_base_oof(name, model_fn, X_train_view, X_test_view, y_train, fold_indices):
    """Verilen model fonksiyonu ile OOF olasılık üret. Tüm base'ler aynı fold_indices kullanır."""
    oof_train = np.zeros(len(X_train_view))
    oof_test = np.zeros(len(X_test_view))
    fold_models = []
    train_f1s = []

    for fold_i, (tr_idx, val_idx) in enumerate(fold_indices):
        X_tr = X_train_view.iloc[tr_idx]
        X_val = X_train_view.iloc[val_idx]
        y_tr = y_train.iloc[tr_idx]
        y_val = y_train.iloc[val_idx]

        model = model_fn(fold_i, X_tr, y_tr, X_val, y_val)

        val_prob = model.predict_proba(X_val)[:, 1]
        test_prob = model.predict_proba(X_test_view)[:, 1]

        oof_train[val_idx] = val_prob
        oof_test += test_prob / len(fold_indices)
        fold_models.append(model)

        train_prob = model.predict_proba(X_tr)[:, 1]
        train_f1 = f1_score(y_tr, (train_prob >= 0.5).astype(int), zero_division=0)
        val_f1 = f1_score(y_val, (val_prob >= 0.5).astype(int), zero_division=0)
        train_f1s.append((train_f1, val_f1))

    avg_train_f1 = np.mean([t[0] for t in train_f1s])
    avg_val_f1 = np.mean([t[1] for t in train_f1s])
    gap = avg_train_f1 - avg_val_f1
    print(f"  [{name}] OOF done | avg train_F1={avg_train_f1:.4f} val_F1={avg_val_f1:.4f} gap={gap:+.4f}")

    return oof_train, oof_test, fold_models, {'train_f1': avg_train_f1, 'val_f1': avg_val_f1, 'gap': gap}


# --- Base 1: LightGBM ---
def lgbm_factory(fold_i, X_tr, y_tr, X_val, y_val):
    model = lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=31, max_depth=8,
        colsample_bytree=0.8, subsample=0.8, reg_alpha=0.1, reg_lambda=1.0,
        class_weight='balanced', random_state=SEED, verbose=-1
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
    return model


# --- Base 2: RandomForest ---
def rf_factory(fold_i, X_tr, y_tr, X_val, y_val):
    model = RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_leaf=3,
        class_weight='balanced', random_state=SEED, n_jobs=-1
    )
    model.fit(X_tr, y_tr)
    return model


# --- Base 3: BalancedBaggingClassifier (LGBM base) ---
def balbag_factory(fold_i, X_tr, y_tr, X_val, y_val):
    base_est = lgb.LGBMClassifier(
        n_estimators=100, learning_rate=0.1, num_leaves=31,
        random_state=SEED, verbose=-1
    )
    model = BalancedBaggingClassifier(
        estimator=base_est, n_estimators=15,
        sampling_strategy='auto', replacement=False,
        random_state=SEED, n_jobs=-1
    )
    model.fit(X_tr, y_tr)
    return model


print("Base model factory'leri hazır (LightGBM, RF, BalancedBagging).")

Base model factory'leri hazır (LightGBM, RF, BalancedBagging).


In [10]:
# Cell 10: NN/DNN Base Modeller (focal loss, F1-based ES, fold-bazlı scaler)

class SmallMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class DNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class NNWrapper:
    """PyTorch NN'i sklearn-uyumlu predict_proba arayüzüne sarar. Fold-bazlı scaler."""

    def __init__(self, model_class, input_dim, hidden_dim=128, dropout=0.4,
                 lr=1e-3, weight_decay=1e-3, epochs=150, batch_size=64,
                 patience=20, focal_gamma=2.0, focal_alpha=0.25):
        self.model_class = model_class
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience
        self.focal_gamma = focal_gamma
        self.focal_alpha = focal_alpha
        self.model_ = None
        self.scaler_ = None

    def _to_tensor(self, X):
        arr = np.array(X, dtype=np.float32)
        arr = np.nan_to_num(arr, nan=0.0)
        return arr

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        X_arr = self._to_tensor(X_train)
        y_arr = np.array(y_train, dtype=np.float32)

        self.scaler_ = StandardScaler()
        X_scaled = self.scaler_.fit_transform(X_arr)

        X_t = torch.FloatTensor(X_scaled)
        y_t = torch.FloatTensor(y_arr)

        self.model_ = self.model_class(self.input_dim, self.hidden_dim, self.dropout)
        criterion = FocalLoss(alpha=self.focal_alpha, gamma=self.focal_gamma)
        optimizer = torch.optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)

        dataset = TensorDataset(X_t, y_t)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        # F1-based early stopping
        best_val_f1 = -1
        patience_cnt = 0
        best_state = None

        if X_val is not None:
            X_val_arr = self.scaler_.transform(self._to_tensor(X_val))
            X_val_t = torch.FloatTensor(X_val_arr)
            y_val_arr = np.array(y_val, dtype=np.float32)

        for epoch in range(self.epochs):
            self.model_.train()
            for bx, by in loader:
                optimizer.zero_grad()
                out = self.model_(bx)
                loss = criterion(out, by)
                loss.backward()
                optimizer.step()

            if X_val is not None:
                self.model_.eval()
                with torch.no_grad():
                    val_logits = self.model_(X_val_t)
                    val_probs = torch.sigmoid(val_logits).numpy()
                val_f1 = f1_score(y_val_arr, (val_probs >= 0.5).astype(int), zero_division=0)
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_state = deepcopy(self.model_.state_dict())
                    patience_cnt = 0
                else:
                    patience_cnt += 1
                    if patience_cnt >= self.patience:
                        break
            else:
                best_state = deepcopy(self.model_.state_dict())

        if best_state is not None:
            self.model_.load_state_dict(best_state)
        return self

    def predict_proba(self, X):
        X_arr = self.scaler_.transform(self._to_tensor(X))
        X_t = torch.FloatTensor(X_arr)
        self.model_.eval()
        with torch.no_grad():
            probs = torch.sigmoid(self.model_(X_t)).numpy()
        return np.column_stack([1 - probs, probs])


# --- Base 4: SmallMLP factory ---
def mlp_factory(fold_i, X_tr, y_tr, X_val, y_val):
    wrapper = NNWrapper(
        model_class=SmallMLP, input_dim=X_tr.shape[1],
        hidden_dim=128, dropout=0.4, lr=1e-3, weight_decay=1e-3,
        epochs=150, batch_size=64, patience=20,
        focal_gamma=2.0, focal_alpha=0.25
    )
    wrapper.fit(X_tr, y_tr, X_val, y_val)
    return wrapper


# --- Base 5: DNN factory ---
def dnn_factory(fold_i, X_tr, y_tr, X_val, y_val):
    wrapper = NNWrapper(
        model_class=DNN, input_dim=X_tr.shape[1],
        hidden_dim=256, dropout=0.5, lr=5e-4, weight_decay=1e-3,
        epochs=200, batch_size=64, patience=25,
        focal_gamma=2.0, focal_alpha=0.25
    )
    wrapper.fit(X_tr, y_tr, X_val, y_val)
    return wrapper


print("NN/DNN factory'leri hazır (focal loss, F1-based ES, fold-bazlı scaler).")

NN/DNN factory'leri hazır (focal loss, F1-based ES, fold-bazlı scaler).


In [11]:
# Cell 11: Tüm Base Modelleri Eğit — OOF Üret
t0 = time.time()
print("=" * 70)
print("5 BASE MODEL EĞİTİMİ BAŞLIYOR (ortak 5-fold OOF)")
print("=" * 70)

base_oof = OrderedDict()
base_stats = {}

# Tree-view base'ler
for name, factory in [('lgbm', lgbm_factory), ('rf', rf_factory), ('balbag', balbag_factory)]:
    oof_tr, oof_te, models, stats = train_base_oof(
        name, factory, X_tr_tree, X_te_tree, y_train, FOLD_INDICES
    )
    base_oof[name] = {'train': oof_tr, 'test': oof_te, 'models': models}
    base_stats[name] = stats

# NN-view base'ler (fold-bazlı scaler NNWrapper içinde)
for name, factory in [('mlp', mlp_factory), ('dnn', dnn_factory)]:
    oof_tr, oof_te, models, stats = train_base_oof(
        name, factory, X_tr_tree, X_te_tree, y_train, FOLD_INDICES
    )
    base_oof[name] = {'train': oof_tr, 'test': oof_te, 'models': models}
    base_stats[name] = stats

elapsed = time.time() - t0
print(f"\nToplam eğitim süresi: {elapsed:.1f}s")
print(f"\nBase stats:")
for name, s in base_stats.items():
    print(f"  {name}: train_F1={s['train_f1']:.4f} val_F1={s['val_f1']:.4f} gap={s['gap']:+.4f}")

5 BASE MODEL EĞİTİMİ BAŞLIYOR (ortak 5-fold OOF)
  [lgbm] OOF done | avg train_F1=0.9631 val_F1=0.8632 gap=+0.0999
  [rf] OOF done | avg train_F1=0.9588 val_F1=0.8646 gap=+0.0941


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [balbag] OOF done | avg train_F1=0.9411 val_F1=0.8443 gap=+0.0967
  [mlp] OOF done | avg train_F1=0.9197 val_F1=0.8498 gap=+0.0699
  [dnn] OOF done | avg train_F1=0.8936 val_F1=0.8469 gap=+0.0467

Toplam eğitim süresi: 48.8s

Base stats:
  lgbm: train_F1=0.9631 val_F1=0.8632 gap=+0.0999
  rf: train_F1=0.9588 val_F1=0.8646 gap=+0.0941
  balbag: train_F1=0.9411 val_F1=0.8443 gap=+0.0967
  mlp: train_F1=0.9197 val_F1=0.8498 gap=+0.0699
  dnn: train_F1=0.8936 val_F1=0.8469 gap=+0.0467


In [12]:
# Cell 12: OOF Korelasyon Matrisi (çeşitlilik kanıtı)
oof_df = pd.DataFrame({name: base_oof[name]['train'] for name in base_oof})
corr_matrix = oof_df.corr(method='pearson')

print("=" * 70)
print("BASE OOF KORELASYON MATRİSİ (Pearson)")
print("=" * 70)
print(corr_matrix.round(4).to_string())

# Ortalama pairwise korelasyon (diagonal hariç)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
pairwise_corrs = corr_matrix.values[mask]
avg_pairwise = np.mean(pairwise_corrs)
min_pairwise = np.min(pairwise_corrs)
max_pairwise = np.max(pairwise_corrs)

print(f"\nOrtalama pairwise korelasyon: {avg_pairwise:.4f}")
print(f"Min: {min_pairwise:.4f} | Max: {max_pairwise:.4f}")
print(f"NB37 referans (GBT-homojen): ~0.90")

if avg_pairwise < 0.70:
    print("→ ÇEŞİTLİLİK KANITI: avg < 0.70, heterojen base BAŞARILI")
elif avg_pairwise < 0.80:
    print("→ ORTA ÇEŞİTLİLİK: avg < 0.80, kısmen dekorele")
else:
    print("→ UYARI: avg ≥ 0.80, çeşitlilik yetersiz olabilir")

# Korelasyon heatmap kaydet
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlBu_r', center=0.5,
            vmin=0, vmax=1, ax=ax, square=True)
ax.set_title('Base OOF Pearson Korelasyon Matrisi')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'oof_correlation_matrix.png'), dpi=150)
plt.close()
print(f"Kaydedildi: {RESULTS_DIR}/oof_correlation_matrix.png")

BASE OOF KORELASYON MATRİSİ (Pearson)
          lgbm      rf  balbag     mlp     dnn
lgbm    1.0000  0.9015  0.9395  0.6411  0.6821
rf      0.9015  1.0000  0.9240  0.6606  0.7228
balbag  0.9395  0.9240  1.0000  0.6471  0.6988
mlp     0.6411  0.6606  0.6471  1.0000  0.7268
dnn     0.6821  0.7228  0.6988  0.7268  1.0000

Ortalama pairwise korelasyon: 0.7544
Min: 0.6411 | Max: 0.9395
NB37 referans (GBT-homojen): ~0.90
→ ORTA ÇEŞİTLİLİK: avg < 0.80, kısmen dekorele
Kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v21_master_diverse_stack_labelshift/oof_correlation_matrix.png


In [13]:
# Cell 13: Tek Model Değerlendirmesi (her base ayrı ayrı — referans çizgi)
print("=" * 70)
print("TEK MODEL DEĞERLENDİRMESİ (base'ler bireysel)")
print("=" * 70)

single_model_results = {}
for name in base_oof:
    proba_te = base_oof[name]['test']
    proba_tr = base_oof[name]['train']
    res = full_eval(name, y_test, proba_te, y_train_for_thr=y_train, proba_train_for_thr=proba_tr)
    # Train metrikleri (overfit gap)
    res_tr = full_eval(f'{name}_TRAIN', y_train, proba_tr, verbose=False)
    res['train_f1'] = res_tr['f1_5050']
    res['train_gap'] = res_tr['f1_5050'] - res['f1_5050']
    single_model_results[name] = res

print("\nEn iyi tek model (%80/20 F1):")
best_single = max(single_model_results, key=lambda k: single_model_results[k]['f1_8020'])
bs = single_model_results[best_single]
print(f"  {best_single}: F1_8020={bs['f1_8020']:.4f} [{bs['f1_8020_ci_lo']:.3f}-{bs['f1_8020_ci_hi']:.3f}]")

TEK MODEL DEĞERLENDİRMESİ (base'ler bireysel)
  [lgbm] thr_8020=0.750 | F1_5050=0.7794 | F1_8020=0.5511 [0.453-0.640]
  [rf] thr_8020=0.710 | F1_5050=0.7926 | F1_8020=0.6087 [0.497-0.696]
  [balbag] thr_8020=0.700 | F1_5050=0.7963 | F1_8020=0.5747 [0.485-0.646]
  [mlp] thr_8020=0.540 | F1_5050=0.8254 | F1_8020=0.5092 [0.448-0.575]
  [dnn] thr_8020=0.550 | F1_5050=0.8075 | F1_8020=0.5188 [0.426-0.590]

En iyi tek model (%80/20 F1):
  rf: F1_8020=0.6087 [0.497-0.696]


In [14]:
# Cell 14: Deney 1 — Heterojen Stacking (Isotonic + LR meta)
print("=" * 70)
print("DENEY 1: HETEROJEN STACKING (5 base → Isotonic → LR meta)")
print("=" * 70)

# Isotonic kalibrasyon (her base OOF train üzerinde fit)
base_calib = {}
iso_regs = {}
for name in base_oof:
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(base_oof[name]['train'], y_train)
    calib_tr = iso.transform(base_oof[name]['train'])
    calib_te = iso.transform(base_oof[name]['test'])
    base_calib[name] = {'train': calib_tr, 'test': calib_te}
    iso_regs[name] = iso

# Meta-features (kalibre OOF)
Xmeta_tr = pd.DataFrame({f'{n}_proba': base_calib[n]['train'] for n in base_calib})
Xmeta_te = pd.DataFrame({f'{n}_proba': base_calib[n]['test'] for n in base_calib})

# Meta-learner: LR (L2, balanced)
meta_lr = LogisticRegression(penalty='l2', C=1.0, max_iter=1000,
                              random_state=SEED, class_weight='balanced')
meta_lr.fit(Xmeta_tr, y_train)

stack_proba_tr = meta_lr.predict_proba(Xmeta_tr)[:, 1]
stack_proba_te = meta_lr.predict_proba(Xmeta_te)[:, 1]

# Değerlendirme
res_stack = full_eval('HetStack_LR', y_test, stack_proba_te,
                       y_train_for_thr=y_train, proba_train_for_thr=stack_proba_tr)
res_stack_tr = full_eval('HetStack_LR_TRAIN', y_train, stack_proba_tr, verbose=False)
res_stack['train_f1'] = res_stack_tr['f1_5050']
res_stack['train_gap'] = res_stack_tr['f1_5050'] - res_stack['f1_5050']

# Meta katsayıları
meta_coefs = dict(zip(list(base_calib.keys()), meta_lr.coef_[0].round(4)))
print(f"\nMeta-learner katsayıları: {meta_coefs}")
print(f"Meta-learner intercept: {meta_lr.intercept_[0]:.4f}")

DENEY 1: HETEROJEN STACKING (5 base → Isotonic → LR meta)
  [HetStack_LR] thr_8020=0.700 | F1_5050=0.8099 | F1_8020=0.6088 [0.517-0.691]

Meta-learner katsayıları: {'lgbm': 1.8461, 'rf': 2.2006, 'balbag': 1.4342, 'mlp': 0.9547, 'dnn': -0.3659}
Meta-learner intercept: -4.1642


In [15]:
# Cell 15: Deney 2 — Calibrate-then-Shift (Saerens kapalı-form)
print("=" * 70)
print("DENEY 2: CALIBRATE-THEN-SHIFT (Saerens kapalı-form düzeltmesi)")
print("=" * 70)

def calibrate_then_shift(proba, pi_train, pi_test):
    """Saerens kapalı-form prior-shift düzeltmesi.
    p_adj(y=1|x) = [pi_test/pi_train * p] / [pi_test/pi_train * p + (1-pi_test)/(1-pi_train) * (1-p)]
    """
    p = np.clip(proba, 1e-8, 1 - 1e-8)
    w1 = pi_test / pi_train
    w0 = (1 - pi_test) / (1 - pi_train)
    p_adj = (w1 * p) / (w1 * p + w0 * (1 - p))
    return p_adj


# V0: Ham olasılık + f1_8020 eşik (baz çizgi — en iyi tek model)
best_single_name = best_single
proba_v0_te = base_oof[best_single_name]['test']
proba_v0_tr = base_oof[best_single_name]['train']

# V1: Kalibrasyon + f1_8020 eşik (sadece kalibrasyon etkisi)
proba_v1_te = base_calib[best_single_name]['test']
proba_v1_tr = base_calib[best_single_name]['train']

# V2: Kalibrasyon + prior-shift + eşik (tam calibrate-then-shift) — tek model üzerinde
proba_v2_te = calibrate_then_shift(proba_v1_te, PI_TRAIN, PI_TEST)
proba_v2_tr = calibrate_then_shift(proba_v1_tr, PI_TRAIN, PI_TEST)

# V3: Stack çıkışına calibrate-then-shift
# Önce stack çıkışını kalibre et (isotonic)
iso_stack = IsotonicRegression(out_of_bounds='clip')
iso_stack.fit(stack_proba_tr, y_train)
stack_calib_tr = iso_stack.transform(stack_proba_tr)
stack_calib_te = iso_stack.transform(stack_proba_te)
proba_v3_te = calibrate_then_shift(stack_calib_te, PI_TRAIN, PI_TEST)
proba_v3_tr = calibrate_then_shift(stack_calib_tr, PI_TRAIN, PI_TEST)

# Değerlendirme
shift_results = {}
for vname, p_te, p_tr in [
    ('V0_ham', proba_v0_te, proba_v0_tr),
    ('V1_calib', proba_v1_te, proba_v1_tr),
    ('V2_calib+shift', proba_v2_te, proba_v2_tr),
    ('V3_stack+shift', proba_v3_te, proba_v3_tr),
]:
    res = full_eval(vname, y_test, p_te, y_train_for_thr=y_train, proba_train_for_thr=p_tr)
    res_tr = full_eval(f'{vname}_TR', y_train, p_tr, verbose=False)
    res['train_f1'] = res_tr['f1_5050']
    res['train_gap'] = res_tr['f1_5050'] - res['f1_5050']
    shift_results[vname] = res

# Kalibrasyon kalitesi (ECE — Expected Calibration Error)
def compute_ece(y_true, y_prob, n_bins=10):
    """Expected Calibration Error."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if mask.sum() == 0:
            continue
        avg_conf = y_prob[mask].mean()
        avg_acc = y_true[mask].mean() if hasattr(y_true, 'mean') else np.mean(y_true[mask])
        ece += mask.sum() / len(y_true) * abs(avg_conf - avg_acc)
    return ece

y_test_arr = y_test.values if hasattr(y_test, 'values') else y_test
ece_v0 = compute_ece(y_test_arr, proba_v0_te)
ece_v1 = compute_ece(y_test_arr, proba_v1_te)
ece_v2 = compute_ece(y_test_arr, proba_v2_te)
ece_stack = compute_ece(y_test_arr, stack_proba_te)
ece_v3 = compute_ece(y_test_arr, proba_v3_te)

print(f"\nKalibrasyon Kalitesi (ECE, düşük=iyi):")
print(f"  V0 (ham):          ECE={ece_v0:.4f}")
print(f"  V1 (calib):        ECE={ece_v1:.4f}")
print(f"  V2 (calib+shift):  ECE={ece_v2:.4f}")
print(f"  Stack (raw):       ECE={ece_stack:.4f}")
print(f"  V3 (stack+shift):  ECE={ece_v3:.4f}")

DENEY 2: CALIBRATE-THEN-SHIFT (Saerens kapalı-form düzeltmesi)
  [V0_ham] thr_8020=0.710 | F1_5050=0.7926 | F1_8020=0.6087 [0.497-0.696]
  [V1_calib] thr_8020=0.790 | F1_5050=0.8021 | F1_8020=0.6165 [0.508-0.699]
  [V2_calib+shift] thr_8020=0.260 | F1_5050=0.8021 | F1_8020=0.6165 [0.508-0.699]
  [V3_stack+shift] thr_8020=0.270 | F1_5050=0.7979 | F1_8020=0.6106 [0.491-0.703]

Kalibrasyon Kalitesi (ECE, düşük=iyi):
  V0 (ham):          ECE=0.0795
  V1 (calib):        ECE=0.0239
  V2 (calib+shift):  ECE=0.3958
  Stack (raw):       ECE=0.1554
  V3 (stack+shift):  ECE=0.3802


In [16]:
# Cell 16: Deney 3 — Raw Probability Meta-Learner (CatBoost / LGBM / LR)
print("=" * 70)
print("DENEY 3: RAW PROBABILITY META-LEARNER (kalibrasyon YOK)")
print("=" * 70)

# Ham OOF olasılıkları → meta-feature
Xraw_tr = pd.DataFrame({f'{n}_raw': base_oof[n]['train'] for n in base_oof})
Xraw_te = pd.DataFrame({f'{n}_raw': base_oof[n]['test'] for n in base_oof})

# Ek meta-features: mean, std, max, min
for df_meta in [Xraw_tr, Xraw_te]:
    vals = df_meta.values
    df_meta['mean_proba'] = vals.mean(axis=1)
    df_meta['std_proba'] = vals.std(axis=1)
    df_meta['max_proba'] = vals.max(axis=1)
    df_meta['min_proba'] = vals.min(axis=1)

print(f"Meta-feature sayısı: {Xraw_tr.shape[1]} (5 base + 4 istatistik)")

raw_meta_results = {}

# Meta 1: Logistic Regression
meta_lr_raw = LogisticRegression(penalty='l2', C=1.0, max_iter=1000,
                                  random_state=SEED, class_weight='balanced')
meta_lr_raw.fit(Xraw_tr, y_train)
lr_proba_tr = meta_lr_raw.predict_proba(Xraw_tr)[:, 1]
lr_proba_te = meta_lr_raw.predict_proba(Xraw_te)[:, 1]
res = full_eval('RawMeta_LR', y_test, lr_proba_te,
                y_train_for_thr=y_train, proba_train_for_thr=lr_proba_tr)
res_tr = full_eval('RawMeta_LR_TR', y_train, lr_proba_tr, verbose=False)
res['train_f1'] = res_tr['f1_5050']
res['train_gap'] = res_tr['f1_5050'] - res['f1_5050']
raw_meta_results['RawMeta_LR'] = res

# Meta 2: LightGBM
meta_lgbm = lgb.LGBMClassifier(
    n_estimators=100, learning_rate=0.05, num_leaves=15, max_depth=4,
    class_weight='balanced', random_state=SEED, verbose=-1,
    colsample_bytree=0.8, subsample=0.8, reg_alpha=0.5, reg_lambda=1.0,
    min_child_samples=20
)
meta_lgbm.fit(Xraw_tr, y_train)
lgbm_proba_tr = meta_lgbm.predict_proba(Xraw_tr)[:, 1]
lgbm_proba_te = meta_lgbm.predict_proba(Xraw_te)[:, 1]
res = full_eval('RawMeta_LGBM', y_test, lgbm_proba_te,
                y_train_for_thr=y_train, proba_train_for_thr=lgbm_proba_tr)
res_tr = full_eval('RawMeta_LGBM_TR', y_train, lgbm_proba_tr, verbose=False)
res['train_f1'] = res_tr['f1_5050']
res['train_gap'] = res_tr['f1_5050'] - res['f1_5050']
raw_meta_results['RawMeta_LGBM'] = res

# Meta 3: CatBoost
meta_cb = CatBoostClassifier(
    iterations=100, learning_rate=0.05, depth=4,
    random_state=SEED, verbose=False,
    auto_class_weights='Balanced',
    l2_leaf_reg=3.0
)
meta_cb.fit(Xraw_tr, y_train)
cb_proba_tr = meta_cb.predict_proba(Xraw_tr)[:, 1]
cb_proba_te = meta_cb.predict_proba(Xraw_te)[:, 1]
res = full_eval('RawMeta_CatBoost', y_test, cb_proba_te,
                y_train_for_thr=y_train, proba_train_for_thr=cb_proba_tr)
res_tr = full_eval('RawMeta_CatBoost_TR', y_train, cb_proba_tr, verbose=False)
res['train_f1'] = res_tr['f1_5050']
res['train_gap'] = res_tr['f1_5050'] - res['f1_5050']
raw_meta_results['RawMeta_CatBoost'] = res

# Calibrate-then-shift en iyi raw meta üzerinde
best_raw_meta_name = max(raw_meta_results, key=lambda k: raw_meta_results[k]['f1_8020'])
print(f"\nEn iyi raw meta: {best_raw_meta_name}")

# En iyi raw meta'ya shift uygula
if best_raw_meta_name == 'RawMeta_LR':
    best_raw_tr, best_raw_te = lr_proba_tr, lr_proba_te
elif best_raw_meta_name == 'RawMeta_LGBM':
    best_raw_tr, best_raw_te = lgbm_proba_tr, lgbm_proba_te
else:
    best_raw_tr, best_raw_te = cb_proba_tr, cb_proba_te

iso_raw = IsotonicRegression(out_of_bounds='clip')
iso_raw.fit(best_raw_tr, y_train)
raw_calib_tr = iso_raw.transform(best_raw_tr)
raw_calib_te = iso_raw.transform(best_raw_te)
raw_shift_te = calibrate_then_shift(raw_calib_te, PI_TRAIN, PI_TEST)
raw_shift_tr = calibrate_then_shift(raw_calib_tr, PI_TRAIN, PI_TEST)

res = full_eval(f'{best_raw_meta_name}+Shift', y_test, raw_shift_te,
                y_train_for_thr=y_train, proba_train_for_thr=raw_shift_tr)
res_tr = full_eval(f'{best_raw_meta_name}+Shift_TR', y_train, raw_shift_tr, verbose=False)
res['train_f1'] = res_tr['f1_5050']
res['train_gap'] = res_tr['f1_5050'] - res['f1_5050']
raw_meta_results[f'{best_raw_meta_name}+Shift'] = res

print(f"\nMeta-learner feature importance (LGBM):")
fi_meta = pd.DataFrame({'feature': Xraw_tr.columns, 'importance': meta_lgbm.feature_importances_})
fi_meta = fi_meta.sort_values('importance', ascending=False)
print(fi_meta.to_string(index=False))

DENEY 3: RAW PROBABILITY META-LEARNER (kalibrasyon YOK)
Meta-feature sayısı: 9 (5 base + 4 istatistik)
  [RawMeta_LR] thr_8020=0.660 | F1_5050=0.8093 | F1_8020=0.5951 [0.501-0.678]
  [RawMeta_LGBM] thr_8020=0.710 | F1_5050=0.7812 | F1_8020=0.6014 [0.482-0.694]
  [RawMeta_CatBoost] thr_8020=0.660 | F1_5050=0.7974 | F1_8020=0.5941 [0.483-0.688]

En iyi raw meta: RawMeta_LGBM
  [RawMeta_LGBM+Shift] thr_8020=0.300 | F1_5050=0.7812 | F1_8020=0.6014 [0.482-0.694]

Meta-learner feature importance (LGBM):
   feature  importance
   mlp_raw         159
    rf_raw         156
  lgbm_raw         155
 std_proba         135
mean_proba         124
   dnn_raw         122
balbag_raw         120
 max_proba          86
 min_proba          83


In [17]:
# Cell 17: Büyük Karşılaştırma Tablosu (tüm deneyler)
print("=" * 90)
print("KAPSAMLI KARŞILAŞTIRMA TABLOSU")
print("=" * 90)

all_results = OrderedDict()

# Tek modeller
for name, res in single_model_results.items():
    all_results[f'Single_{name}'] = res

# Deney 1: Heterojen stacking
all_results['HetStack_LR'] = res_stack

# Deney 2: Calibrate-then-shift varyantları
for name, res in shift_results.items():
    all_results[name] = res

# Deney 3: Raw meta-learner
for name, res in raw_meta_results.items():
    all_results[name] = res

# NB36 referans
all_results['NB36_BalBag_XGB'] = {
    'name': 'NB36_BalBag_XGB', 'f1_8020': 0.6025, 'f1_8020_ci_lo': None, 'f1_8020_ci_hi': None,
    'f1_5050': 0.6025, 'mcc_8020': None, 'prec_8020': None, 'rec_8020': None,
    'train_f1': None, 'train_gap': None, 'auc_roc': None,
}

rows = []
for name, res in all_results.items():
    ci = ''
    if res.get('f1_8020_ci_lo') is not None:
        ci = f"[{res['f1_8020_ci_lo']:.3f}-{res['f1_8020_ci_hi']:.3f}]"
    rows.append({
        'Model': name,
        'F1_8020': round(res.get('f1_8020', 0), 4),
        'CI_95': ci,
        'MCC_8020': round(res.get('mcc_8020', 0) or 0, 4),
        'Prec_8020': round(res.get('prec_8020', 0) or 0, 4),
        'Rec_8020': round(res.get('rec_8020', 0) or 0, 4),
        'F1_5050': round(res.get('f1_5050', 0) or 0, 4),
        'AUC': round(res.get('auc_roc', 0) or 0, 4),
        'Gap': round(res.get('train_gap', 0) or 0, 4),
    })

comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df.sort_values('F1_8020', ascending=False)
print(comparison_df.to_string(index=False))

comparison_df.to_csv(os.path.join(RESULTS_DIR, 'full_comparison.csv'), index=False)

# En iyi genel
best_overall = comparison_df.iloc[0]
print(f"\n→ EN İYİ MODEL: {best_overall['Model']} | F1_8020={best_overall['F1_8020']:.4f} {best_overall['CI_95']}")
print(f"→ NB36 referans: 0.6025")
improvement = best_overall['F1_8020'] - 0.6025
print(f"→ İyileştirme: {improvement:+.4f}")

KAPSAMLI KARŞILAŞTIRMA TABLOSU
             Model  F1_8020         CI_95  MCC_8020  Prec_8020  Rec_8020  F1_5050    AUC     Gap
          V1_calib   0.6165 [0.508-0.699]    0.5142     0.5497    0.7062   0.8021 0.8341 -0.0212
    V2_calib+shift   0.6165 [0.508-0.699]    0.5142     0.5497    0.7062   0.8021 0.8341 -0.0212
    V3_stack+shift   0.6106 [0.491-0.703]    0.5064     0.5392    0.7087   0.7979 0.8477 -0.0200
       HetStack_LR   0.6088 [0.517-0.691]    0.5040     0.5236    0.7323   0.8099 0.8439 -0.0258
         Single_rf   0.6087 [0.497-0.696]    0.5042     0.5453    0.6933   0.7926 0.8389 -0.0159
            V0_ham   0.6087 [0.497-0.696]    0.5042     0.5453    0.6933   0.7926 0.8389 -0.0159
   NB36_BalBag_XGB   0.6025                  0.0000     0.0000    0.0000   0.6025 0.0000  0.0000
      RawMeta_LGBM   0.6014 [0.482-0.694]    0.4951     0.5423    0.6800   0.7812 0.8434 -0.0002
RawMeta_LGBM+Shift   0.6014 [0.482-0.694]    0.4951     0.5423    0.6800   0.7812 0.8439 -0.0012

In [18]:
# Cell 18: is_missing_* Flag Importance ve Leakage Uyarısı
print("=" * 70)
print("LEAKAGE PROBE: is_missing_* Flag Importance")
print("=" * 70)

miss_flags = [c for c in X_tr_tree.columns if str(c).startswith('is_missing_')]
fi_model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED, verbose=-1)
fi_model.fit(X_tr_tree, y_train)
fi_df = pd.DataFrame({'feature': X_tr_tree.columns, 'importance': fi_model.feature_importances_})
total_fi = fi_df['importance'].sum()

if miss_flags and total_fi > 0:
    mfi = fi_df[fi_df['feature'].isin(miss_flags)]
    miss_fi_pct = mfi['importance'].sum() / total_fi * 100
    top_miss = mfi.sort_values('importance', ascending=False).head(10)
    print(f"is_missing_* toplam importance: {miss_fi_pct:.1f}%")
    print(f"Top 10 is_missing flag:")
    for _, row in top_miss.iterrows():
        pct = row['importance'] / total_fi * 100
        print(f"  {row['feature']}: {pct:.2f}%")
    if miss_fi_pct > 15:
        print("→ UYARI: is_missing importance > %15 — leakage riski yüksek")
    else:
        print("→ OK: is_missing importance < %15 — leakage riski düşük")
else:
    miss_fi_pct = 0.0
    print("is_missing flag bulunamadı veya importance hesaplanamadı.")

LEAKAGE PROBE: is_missing_* Flag Importance
is_missing_* toplam importance: 0.9%
Top 10 is_missing flag:
  is_missing_AL_16: 0.33%
  is_missing_AL_1: 0.30%
  is_missing_AL_183: 0.23%
  is_missing_AL_214: 0.03%
  is_missing_AL_239: 0.00%
  is_missing_AL_232: 0.00%
  is_missing_AL_234: 0.00%
  is_missing_AL_235: 0.00%
  is_missing_AL_237: 0.00%
  is_missing_AL_238: 0.00%
→ OK: is_missing importance < %15 — leakage riski düşük


In [19]:
# Cell 19: Görselleştirmeler
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1. Tüm modellerin F1_8020 karşılaştırması
plot_df = comparison_df[comparison_df['Model'] != 'NB36_BalBag_XGB'].copy()
colors = []
for m in plot_df['Model']:
    if 'Single' in m: colors.append('#3498db')
    elif 'HetStack' in m: colors.append('#e74c3c')
    elif 'RawMeta' in m: colors.append('#2ecc71')
    else: colors.append('#f39c12')
ax = axes[0, 0]
bars = ax.barh(range(len(plot_df)), plot_df['F1_8020'].values, color=colors)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['Model'].values, fontsize=7)
ax.axvline(0.6025, color='red', linestyle='--', label='NB36 ref (0.6025)')
ax.set_xlabel('F1_8020')
ax.set_title('Tüm Modeller — %80/20 F1')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3, axis='x')

# 2. Korelasyon heatmap (küçük)
ax = axes[0, 1]
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0.5,
            vmin=0, vmax=1, ax=ax, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Base OOF Korelasyon')

# 3. Calibrate-then-shift ablasyon
ax = axes[0, 2]
shift_names = list(shift_results.keys())
shift_f1s = [shift_results[n]['f1_8020'] for n in shift_names]
shift_colors = ['#bdc3c7', '#3498db', '#e74c3c', '#9b59b6']
ax.bar(range(len(shift_names)), shift_f1s, color=shift_colors[:len(shift_names)])
ax.set_xticks(range(len(shift_names)))
ax.set_xticklabels(shift_names, rotation=30, ha='right', fontsize=7)
ax.set_ylabel('F1_8020')
ax.set_title('Calibrate-then-Shift Ablasyon')
ax.grid(True, alpha=0.3, axis='y')

# 4. Raw meta-learner karşılaştırma
ax = axes[1, 0]
raw_names = list(raw_meta_results.keys())
raw_f1s = [raw_meta_results[n]['f1_8020'] for n in raw_names]
ax.bar(range(len(raw_names)), raw_f1s, color=['#2ecc71', '#27ae60', '#1abc9c', '#16a085'][:len(raw_names)])
ax.set_xticks(range(len(raw_names)))
ax.set_xticklabels(raw_names, rotation=30, ha='right', fontsize=7)
ax.set_ylabel('F1_8020')
ax.set_title('Raw Meta-Learner Karşılaştırma')
ax.axhline(0.6025, color='red', linestyle='--', label='NB36 ref')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3, axis='y')

# 5. Train-test gap (overfit kontrolü)
ax = axes[1, 1]
gap_names = [n for n in all_results if all_results[n].get('train_gap') is not None]
gaps = [all_results[n]['train_gap'] for n in gap_names]
gap_colors = ['red' if g > 0.15 else 'orange' if g > 0.10 else 'green' for g in gaps]
ax.barh(range(len(gap_names)), gaps, color=gap_colors)
ax.set_yticks(range(len(gap_names)))
ax.set_yticklabels(gap_names, fontsize=7)
ax.axvline(0.15, color='red', linestyle='--', alpha=0.5, label='Overfit alarm (0.15)')
ax.set_xlabel('Train-Test F1 Gap')
ax.set_title('Overfit Kontrolü')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3, axis='x')

# 6. Bootstrap dağılımı (en iyi model)
ax = axes[1, 2]
best_name = comparison_df.iloc[0]['Model']
if best_name in all_results and 'f1_8020_scores' in all_results[best_name]:
    scores = all_results[best_name]['f1_8020_scores']
    ax.hist(scores, bins=20, edgecolor='black', alpha=0.7, color='#e74c3c')
    ax.axvline(np.mean(scores), color='blue', linestyle='--', label=f'Mean: {np.mean(scores):.3f}')
    ax.axvline(np.percentile(scores, 2.5), color='green', linestyle=':', label='95% CI')
    ax.axvline(np.percentile(scores, 97.5), color='green', linestyle=':')
    ax.set_xlabel('F1')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Bootstrap F1 — {best_name}')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'all_experiments.png'), dpi=150)
plt.close()
print(f"Görselleştirmeler kaydedildi: {RESULTS_DIR}/all_experiments.png")

Görselleştirmeler kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v21_master_diverse_stack_labelshift/all_experiments.png


In [20]:
# Cell 20: Reliability Diagram (Kalibrasyon Kalitesi)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def reliability_diagram(ax, y_true, y_prob, title, n_bins=10):
    """Reliability diagram çiz."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_centers = []
    bin_means = []
    bin_counts = []
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if mask.sum() == 0:
            continue
        bin_centers.append(y_prob[mask].mean())
        y_arr = y_true[mask] if isinstance(y_true, np.ndarray) else y_true.values[mask]
        bin_means.append(y_arr.mean())
        bin_counts.append(mask.sum())
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
    ax.scatter(bin_centers, bin_means, s=[c*2 for c in bin_counts], alpha=0.7, zorder=5)
    ax.plot(bin_centers, bin_means, 'o-', alpha=0.7)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(title)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.legend()
    ax.grid(True, alpha=0.3)

reliability_diagram(axes[0], y_test, proba_v0_te, f'V0 Ham ({best_single}) ECE={ece_v0:.3f}')
reliability_diagram(axes[1], y_test, proba_v1_te, f'V1 Isotonic Calib ECE={ece_v1:.3f}')
reliability_diagram(axes[2], y_test, proba_v2_te, f'V2 Calib+Shift ECE={ece_v2:.3f}')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'reliability_diagrams.png'), dpi=150)
plt.close()
print(f"Kaydedildi: {RESULTS_DIR}/reliability_diagrams.png")

Kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v21_master_diverse_stack_labelshift/reliability_diagrams.png


In [21]:
# Cell 21: Kapsamlı PDF Rapor

class NB38Report(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.cell(0, 10, 'NB38: MASTER - Heterojen Stacking + Calibrate-then-Shift', 0, 1, 'C')
        self.set_font('Helvetica', '', 9)
        self.cell(0, 5, f'SEED={SEED} | TEST_SIZE={TEST_SIZE} | 5 base aile | 3 deney', 0, 1, 'C')
        self.ln(3)

    def section_title(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.set_fill_color(41, 128, 185)
        self.set_text_color(255, 255, 255)
        self.cell(0, 8, f'  {title}', 0, 1, 'L', fill=True)
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def body_text(self, text):
        self.set_font('Helvetica', '', 9)
        self.multi_cell(0, 5, text)
        self.ln(2)

    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [self.epw / len(headers)] * len(headers)
        self.set_font('Helvetica', 'B', 7)
        self.set_fill_color(52, 73, 94)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), 1, 0, 'C', fill=True)
        self.ln()
        self.set_font('Helvetica', '', 6.5)
        self.set_text_color(0, 0, 0)
        for j, row in enumerate(rows):
            if j % 2 == 0:
                self.set_fill_color(236, 240, 241)
            else:
                self.set_fill_color(255, 255, 255)
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, 'C', fill=True)
            self.ln()
        self.ln(3)


pdf = NB38Report()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# --- 0. Motivasyon ve Amaç ---
pdf.section_title('0. Motivasyon ve Amac')
pdf.body_text(
    'NB37 stacking basarisiz oldu (best arm 0.589 < BalancedBag_XGB 0.6025). '
    'Kok neden: tum base modeller GBT ailesinden (LGBM/XGB/CatBoost/BalBag_XGB), '
    'OOF tahminleri ~0.9 korele. Meta-learner korele tahminlerden ek bilgi cikaramadi.\n\n'
    'NB38 uc deney ile bu sorunu cozer:\n'
    '(1) Heterojen Stacking: 5 farkli aile (LGBM, RF, BalBag, SmallMLP, DNN) ile '
    'OOF korelasyonu dusurulur, Isotonic kalibrasyon + LR meta.\n'
    '(2) Calibrate-then-Shift: Saerens kapali-form prior-shift duzeltmesi '
    '(pi_train=0.733 -> pi_test=0.20). Kalibrasyon on-kosul.\n'
    '(3) Raw Probability Meta-Learner: Ham OOF olasiliklarini CatBoost/LGBM/LR meta-learner a besle.'
)

# --- 1. Veri ---
pdf.section_title('1. Veri ve Split')
pdf.body_text(
    f'Veri: YARISMA_TRAIN_MASTER.csv (n={len(df)}, pos={int(y.sum())}, neg={int((y==0).sum())})\n'
    f'Split: %80/20 stratified (SEED={SEED})\n'
    f'Train: {len(y_train)} | Test: {len(y_test)}\n'
    f'Feature temizligi: sabit={len(const_cols)}, ozdes cift={len(dup_col_pairs)}, '
    f'toplam drop={len(drop_cols)}, kalan={X_tr_tree.shape[1]}\n'
    f'Missing strateji: M3 (is_missing flag >%50 + medyan imputation)'
)

# --- 2. OOF Korelasyon Matrisi ---
pdf.section_title('2. Base OOF Korelasyon Matrisi (Cesitlilik Kaniti)')
corr_headers = [''] + list(corr_matrix.columns)
corr_rows = []
for name in corr_matrix.index:
    row = [name] + [f'{corr_matrix.loc[name, c]:.3f}' for c in corr_matrix.columns]
    corr_rows.append(row)
n_corr_cols = len(corr_headers)
cw = [190 / n_corr_cols] * n_corr_cols
pdf.add_table(corr_headers, corr_rows, cw)
pdf.body_text(
    f'Ortalama pairwise korelasyon: {avg_pairwise:.4f} '
    f'(NB37 referans: ~0.90)\n'
    f'Min: {min_pairwise:.4f} | Max: {max_pairwise:.4f}'
)

# --- 3. Base Model Train-Test Gap ---
pdf.section_title('3. Base Modeller - Train/Test Gap (Overfit Kontrolu)')
gap_headers = ['Base', 'Train F1', 'Val F1', 'Gap']
gap_rows = []
for name, s in base_stats.items():
    gap_rows.append([name, f"{s['train_f1']:.4f}", f"{s['val_f1']:.4f}", f"{s['gap']:+.4f}"])
pdf.add_table(gap_headers, gap_rows, [40, 40, 40, 40])

# --- 4. Tek Model Sonuçları ---
pdf.section_title('4. Tek Model Sonuclari')
sm_headers = ['Model', 'F1_8020', 'CI_95', 'MCC_8020', 'F1_5050', 'AUC', 'Gap']
sm_rows = []
for name, res in single_model_results.items():
    ci = f"[{res['f1_8020_ci_lo']:.3f}-{res['f1_8020_ci_hi']:.3f}]"
    sm_rows.append([
        name, f"{res['f1_8020']:.4f}", ci,
        f"{res.get('mcc_8020',0):.4f}", f"{res['f1_5050']:.4f}",
        f"{res.get('auc_roc',0):.4f}", f"{res.get('train_gap',0):+.4f}"
    ])
pdf.add_table(sm_headers, sm_rows, [25, 22, 35, 25, 22, 22, 22])

# --- 5. Deney 1: Heterojen Stacking ---
pdf.section_title('5. Deney 1: Heterojen Stacking (Isotonic + LR Meta)')
ci_stack = f"[{res_stack['f1_8020_ci_lo']:.3f}-{res_stack['f1_8020_ci_hi']:.3f}]"
pdf.body_text(
    f"Sonuc: F1_8020={res_stack['f1_8020']:.4f} {ci_stack}\n"
    f"MCC_8020={res_stack.get('mcc_8020',0):.4f} | F1_5050={res_stack['f1_5050']:.4f}\n"
    f"Train gap={res_stack.get('train_gap',0):+.4f}\n\n"
    f"Meta katsayilari: {meta_coefs}\n\n"
    f"NB36 referans: 0.6025 | Fark: {res_stack['f1_8020'] - 0.6025:+.4f}"
)

# --- 6. Deney 2: Calibrate-then-Shift ---
pdf.section_title('6. Deney 2: Calibrate-then-Shift Ablasyon')
sh_headers = ['Varyant', 'F1_8020', 'CI_95', 'MCC_8020', 'ECE', 'Gap']
ece_map = {'V0_ham': ece_v0, 'V1_calib': ece_v1, 'V2_calib+shift': ece_v2, 'V3_stack+shift': ece_v3}
sh_rows = []
for name, res in shift_results.items():
    ci = f"[{res['f1_8020_ci_lo']:.3f}-{res['f1_8020_ci_hi']:.3f}]"
    sh_rows.append([
        name, f"{res['f1_8020']:.4f}", ci,
        f"{res.get('mcc_8020',0):.4f}",
        f"{ece_map.get(name, 0):.4f}",
        f"{res.get('train_gap',0):+.4f}"
    ])
pdf.add_table(sh_headers, sh_rows, [35, 22, 38, 25, 22, 22])
pdf.body_text(
    'Saerens kapali-form: p_adj = [pi_test/pi_train * p] / '
    '[pi_test/pi_train * p + (1-pi_test)/(1-pi_train) * (1-p)]\n'
    f'pi_train={PI_TRAIN}, pi_test={PI_TEST}\n\n'
    'Kalibrasyon on-kosul: Isotonic regression OOF uzerinde fit. '
    'ECE dusuk = iyi kalibre. Shift sonrasi ECE artabilir (beklenen, prior degisti).'
)

# --- 7. Deney 3: Raw Meta-Learner ---
pdf.section_title('7. Deney 3: Raw Probability Meta-Learner')
rm_headers = ['Meta', 'F1_8020', 'CI_95', 'MCC_8020', 'F1_5050', 'Gap']
rm_rows = []
for name, res in raw_meta_results.items():
    ci = f"[{res['f1_8020_ci_lo']:.3f}-{res['f1_8020_ci_hi']:.3f}]"
    rm_rows.append([
        name, f"{res['f1_8020']:.4f}", ci,
        f"{res.get('mcc_8020',0):.4f}", f"{res['f1_5050']:.4f}",
        f"{res.get('train_gap',0):+.4f}"
    ])
pdf.add_table(rm_headers, rm_rows, [35, 22, 38, 25, 25, 22])
pdf.body_text(
    'Raw meta-learner: Isotonic kalibrasyon ATLANIR, ham OOF olasiliklarini dogrudan '
    'meta-learner a besler. Ek meta-feature: mean, std, max, min proba.\n'
    f'Meta-feature sayisi: {Xraw_tr.shape[1]}\n\n'
    f'En iyi raw meta: {best_raw_meta_name}'
)

# --- 8. Leakage ---
pdf.section_title('8. Leakage Probe: is_missing Flag Importance')
pdf.body_text(
    f'is_missing_* toplam importance: {miss_fi_pct:.1f}%\n'
    f'{"UYARI: > %15 esik" if miss_fi_pct > 15 else "OK: < %15 esik"}'
)

# --- 9. Büyük Karşılaştırma ---
pdf.add_page()
pdf.section_title('9. Buyuk Karsilastirma Tablosu')
comp_headers = ['Model', 'F1_8020', 'CI_95', 'MCC', 'Prec', 'Rec', 'F1_50', 'AUC', 'Gap']
comp_rows = []
for _, row in comparison_df.iterrows():
    comp_rows.append([
        row['Model'][:22], f"{row['F1_8020']:.4f}", row['CI_95'][:18],
        f"{row['MCC_8020']:.3f}", f"{row['Prec_8020']:.3f}",
        f"{row['Rec_8020']:.3f}", f"{row['F1_5050']:.3f}",
        f"{row['AUC']:.3f}", f"{row['Gap']:+.3f}"
    ])
pdf.add_table(comp_headers, comp_rows, [38, 18, 30, 18, 18, 18, 16, 16, 18])

# --- 10. Sonuç ve Öneriler ---
pdf.section_title('10. Sonuc ve Oneriler')
best_name = comparison_df.iloc[0]['Model']
best_f1 = comparison_df.iloc[0]['F1_8020']
improvement = best_f1 - 0.6025
pdf.body_text(
    f'En iyi model: {best_name} (F1_8020={best_f1:.4f})\n'
    f'NB36 referans: 0.6025 | Iyilestirme: {improvement:+.4f}\n\n'
    'Cesitlilik kaniti:\n'
    f'  - Ortalama pairwise OOF korelasyon: {avg_pairwise:.4f} (NB37 ~0.90)\n'
    f'  - Min pairwise: {min_pairwise:.4f}\n\n'
)

if improvement > 0.01:
    pdf.body_text(
        f'SONUC: Heterojen stacking + calibrate-then-shift MASTER da tek modeli '
        f'{improvement:+.4f} ile gecti. Teslim adayi: {best_name}.'
    )
elif improvement > -0.005:
    pdf.body_text(
        'SONUC: Stacking/shift tek modelle esit veya marjinal fark. '
        'MASTER da cesitli base ile bile tek modeli gecilemedi olabilir. '
        'Bu gecerli bir sonuc — teslimde en iyi tek model kullanilabilir.'
    )
else:
    pdf.body_text(
        'SONUC: Stacking/shift tek modelin altinda kaldi. MASTER da ek sinyal '
        'guclu bir GBT tarafindan buyuk olcude yakalaniyor. Teslimde tek model kullan.'
    )

# Görseller ekle
for img_name in ['all_experiments.png', 'oof_correlation_matrix.png', 'reliability_diagrams.png']:
    img_path = os.path.join(RESULTS_DIR, img_name)
    if os.path.exists(img_path):
        pdf.add_page()
        pdf.section_title(f'Gorsel: {img_name}')
        pdf.image(img_path, x=10, w=190)

# Kaydet
report_path = os.path.join(REPORTS_DIR, 'NB38_master_diverse_stack_labelshift_report.pdf')
pdf.output(report_path)
print(f"PDF rapor kaydedildi: {report_path}")

PDF rapor kaydedildi: /Users/tefe/teknofest_model/teknofest_model/reports/NB38_master_diverse_stack_labelshift_report.pdf


In [22]:
# Cell 22: Sonuçları Kaydet (JSON)

def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_serializable(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (pd.Series, pd.DataFrame)):
        return str(obj)
    elif hasattr(obj, '__dict__'):
        return str(type(obj).__name__)
    return obj

save_data = {
    'metadata': {
        'notebook': 'NB38', 'seed': SEED, 'test_size': TEST_SIZE,
        'pi_train': PI_TRAIN, 'pi_test': PI_TEST,
        'n_bootstrap': N_BOOTSTRAP, 'n_folds': N_FOLDS,
        'n_features': X_tr_tree.shape[1],
    },
    'correlation': {
        'matrix': corr_matrix.to_dict(),
        'avg_pairwise': avg_pairwise,
        'min': min_pairwise, 'max': max_pairwise,
    },
    'base_stats': base_stats,
    'single_model_results': {k: {kk: make_serializable(vv) for kk, vv in v.items() if kk != 'proba'}
                             for k, v in single_model_results.items()},
    'het_stack': {k: make_serializable(v) for k, v in res_stack.items() if k != 'proba'},
    'shift_results': {k: {kk: make_serializable(vv) for kk, vv in v.items() if kk != 'proba'}
                      for k, v in shift_results.items()},
    'raw_meta_results': {k: {kk: make_serializable(vv) for kk, vv in v.items() if kk != 'proba'}
                         for k, v in raw_meta_results.items()},
    'ece': {'v0': ece_v0, 'v1': ece_v1, 'v2': ece_v2, 'stack': ece_stack, 'v3': ece_v3},
    'meta_coefs': meta_coefs,
    'miss_fi_pct': miss_fi_pct,
}

with open(os.path.join(RESULTS_DIR, 'nb38_results.json'), 'w') as f:
    json.dump(make_serializable(save_data), f, indent=2)

print(f"Sonuçlar kaydedildi: {RESULTS_DIR}/nb38_results.json")

Sonuçlar kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v21_master_diverse_stack_labelshift/nb38_results.json


In [23]:
# Cell 23: Final Özet
print("\n" + "=" * 70)
print("NB38 TAMAMLANDI: MASTER Panel — Heterojen Stacking + Calibrate-then-Shift")
print("=" * 70)

print(f"\n{'Deney':<35} {'F1_8020':>10} {'vs NB36':>10}")
print("-" * 60)

# Tek modeller
for name, res in single_model_results.items():
    delta = res['f1_8020'] - 0.6025
    print(f"Single_{name:<28} {res['f1_8020']:>10.4f} {delta:>+10.4f}")

# Heterojen stack
delta = res_stack['f1_8020'] - 0.6025
print(f"{'HetStack_LR':<35} {res_stack['f1_8020']:>10.4f} {delta:>+10.4f}")

# Shift varyantları
for name, res in shift_results.items():
    delta = res['f1_8020'] - 0.6025
    print(f"{name:<35} {res['f1_8020']:>10.4f} {delta:>+10.4f}")

# Raw meta
for name, res in raw_meta_results.items():
    delta = res['f1_8020'] - 0.6025
    print(f"{name:<35} {res['f1_8020']:>10.4f} {delta:>+10.4f}")

print(f"\n{'NB36 referans':<35} {'0.6025':>10}")
print(f"\n→ Korelasyon: avg={avg_pairwise:.4f} (NB37 ~0.90)")
print(f"→ Çıktı: {RESULTS_DIR}")
print(f"→ Rapor: {os.path.join(REPORTS_DIR, 'NB38_master_diverse_stack_labelshift_report.pdf')}")


NB38 TAMAMLANDI: MASTER Panel — Heterojen Stacking + Calibrate-then-Shift

Deney                                  F1_8020    vs NB36
------------------------------------------------------------
Single_lgbm                             0.5511    -0.0514
Single_rf                               0.6087    +0.0062
Single_balbag                           0.5747    -0.0278
Single_mlp                              0.5092    -0.0933
Single_dnn                              0.5188    -0.0837
HetStack_LR                             0.6088    +0.0063
V0_ham                                  0.6087    +0.0062
V1_calib                                0.6165    +0.0140
V2_calib+shift                          0.6165    +0.0140
V3_stack+shift                          0.6106    +0.0081
RawMeta_LR                              0.5951    -0.0074
RawMeta_LGBM                            0.6014    -0.0011
RawMeta_CatBoost                        0.5941    -0.0084
RawMeta_LGBM+Shift                      0.6014    -